In [ ]:
import matplotlib
# Force Matplotlib to use the standard windowing backend for animation
matplotlib.use('TkAgg') 

import cv2
import json
import requests
import numpy as np
import math
import matplotlib.pyplot as plt
from pupil_apriltags import Detector
import os

print("✅ Libraries imported and backend configured.")

In [ ]:
def calculate_shot_angle_optimized(
    v0: float,
    distance: float,
    target_height: float = 2.64,   # 2026 Hub Height
    muzzle_height: float = 0.508,  # Robot shooter height
    ball_properties = None,
    env_properties = None,
    use_drag: bool = True,
    arc: str = "low",              
    dt: float = 0.01,              
    coarse_step_deg: float = 1.0,  
    tol_m: float = 0.01,           
    max_time_s: float = 5.0          
    ):

    # --- Defaults & Constants ---
    if ball_properties is None:
        ball_properties = {"mass": 0.215, "radius": 0.075, "drag_coeff": 0.5}
    if env_properties is None:
        env_properties = {"gravity": 9.81, "air_density": 1.225}

    g = env_properties["gravity"]
    rho = env_properties["air_density"]
    m = ball_properties["mass"]
    r = ball_properties["radius"]
    Cd = ball_properties["drag_coeff"]
    area = math.pi * r * r
    k = 0.5 * rho * Cd * area 

    dx = float(distance)
    dy = float(target_height - muzzle_height)

    # Sanity check
    if v0 <= 0 or dx <= 0:
        return None

    # --- Fast Vacuum Check (Analytic) ---
    v_sq = v0 * v0
    term1 = (g * dx * dx) / (2.0 * v_sq)
    disc = (-dx)**2 - 4.0 * term1 * (dy + term1)

    if disc < 0:
        return None # Unreachable

    # --- RK2 Flight Simulation ---
    def simulate_y_at_dx(angle_rad: float):
        x, y = 0.0, 0.0
        vx = v0 * math.cos(angle_rad)
        vy = v0 * math.sin(angle_rad)
        t = 0.0
        
        while t < max_time_s:
            v = math.hypot(vx, vy)
            if v < 1e-1: return None 

            fd = k * v * v
            ax = -(fd * (vx / v)) / m
            ay = -g - (fd * (vy / v)) / m
            
            # RK2 Step
            x += vx * dt
            y += vy * dt
            vx += ax * dt
            vy += ay * dt
            t += dt

            if x >= dx:
                return y

        return None

    def err(angle_rad):
        y_at = simulate_y_at_dx(angle_rad)
        if y_at is None: return None
        return y_at - dy

    # --- Coarse Scan ---
    brackets = []
    prev_a = math.radians(1.0)
    prev_e = err(prev_a)

    for i in range(2, 85):
        a = math.radians(float(i))
        e = err(a)
        if prev_e is not None and e is not None:
            if prev_e * e <= 0:
                brackets.append((prev_a, a))
        prev_a, prev_e = a, e

    if not brackets:
        return None

    # --- Bisection Solver (Low Arc Only) ---
    lo, hi = brackets[0]
    for _ in range(20):
        mid = 0.5 * (lo + hi)
        em = err(mid)
        if em is None: return None
        if abs(em) < tol_m: return math.degrees(mid)
        
        if err(lo) * em < 0:
            hi = mid
        else:
            lo = mid
            
    return math.degrees(0.5 * (lo + hi))

In [ ]:
class FieldMapper:
    def __init__(self, json_path):
        self.tag_poses = {} 
        self.field_length = 17.55
        self.field_width = 8.05
        self.load_json(json_path)

    def load_json(self, path):
        if not os.path.exists(path):
            print(f"File {path} not found. Please download it first.")
            return

        with open(path, 'r') as f:
            data = json.load(f)
        
        if 'field' in data:
            self.field_length = data['field'].get('length', 17.55)
            self.field_width = data['field'].get('width', 8.05)

        for tag in data['tags']:
            id = tag['ID']
            tx = tag['pose']['translation']['x']
            ty = tag['pose']['translation']['y']
            tz = tag['pose']['translation']['z']
            qw = tag['pose']['rotation']['quaternion']['W']
            qx = tag['pose']['rotation']['quaternion']['X']
            qy = tag['pose']['rotation']['quaternion']['Y']
            qz = tag['pose']['rotation']['quaternion']['Z']
            self.tag_poses[id] = self.create_matrix(tx, ty, tz, qw, qx, qy, qz)
            
    def create_matrix(self, tx, ty, tz, qw, qx, qy, qz):
        xx, yy, zz = qx*qx, qy*qy, qz*qz
        xy, xz, yz = qx*qy, qx*qz, qy*qz
        wx, wy, wz = qw*qx, qw*qy, qw*qz

        R = np.array([
            [1 - 2*(yy+zz), 2*(xy-wz),   2*(xz+wy)],
            [2*(xy+wz),     1 - 2*(xx+zz), 2*(yz-wx)],
            [2*(xz-wy),     2*(yz+wx),     1 - 2*(xx+yy)]
        ])
        
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = [tx, ty, tz]
        return T

    def get_cam_position(self, tag_id, r_matrix, t_vec):
        if tag_id not in self.tag_poses:
            return None
        
        # 1. Optical Transform
        T_cam_opt_to_tag_opt = np.eye(4)
        T_cam_opt_to_tag_opt[:3, :3] = r_matrix
        T_cam_opt_to_tag_opt[:3, 3] = t_vec.flatten()
        
        # 2. Basis Change (Optical -> NWU)
        M_opt_to_nwu = np.array([
            [0, 0, 1, 0],   # Z (Forward) -> X
            [-1, 0, 0, 0],  # -X (Left)   -> Y
            [0, -1, 0, 0],  # -Y (Up)     -> Z
            [0, 0, 0, 1]
        ])
        
        M_inv = np.linalg.inv(M_opt_to_nwu)
        T_cam_nwu_to_tag_nwu = M_opt_to_nwu @ T_cam_opt_to_tag_opt @ M_inv
        
        # 3. Invert to get Tag -> Cam
        T_tag_nwu_to_cam_nwu = np.linalg.inv(T_cam_nwu_to_tag_nwu)

        # --- FIX 1: INVERT TRANSLATION (Kept from before) ---
        T_tag_nwu_to_cam_nwu[:3, 3] = -T_tag_nwu_to_cam_nwu[:3, 3]

        # 4. Chain to Field
        T_field_to_tag = self.tag_poses[tag_id]
        T_field_to_cam = T_field_to_tag @ T_tag_nwu_to_cam_nwu
        
        # --- FIX 2: FLIP HEADING 180 DEGREES ---
        # Calculate yaw and add PI radians (180 degrees) to point the arrow 
        # in the direction the camera is looking.
        yaw = np.arctan2(T_field_to_cam[1, 0], T_field_to_cam[0, 0]) + np.pi
        
        return T_field_to_cam[0, 3], T_field_to_cam[1, 3], T_field_to_cam[2, 3], yaw

# Geometry Helper
def get_intersection(p1, p2, p3, p4):
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    x4, y4 = p4
    denom = (x1 - x2) * (y3 - y4) - (y1 - y2) * (x3 - x4)
    if denom == 0: return None 
    px = ((x1*y2 - y1*x2)*(x3 - x4) - (x1 - x2)*(x3*y4 - y3*x4)) / denom
    py = ((x1*y2 - y1*x2)*(y3 - y4) - (y1 - y2)*(x3*y4 - y3*x4)) / denom
    return (px, py)

In [ ]:
# --- Configuration ---
JSON_FILENAME = "2026-rebuilt-welded.json"
TAG_SIZE = 0.1651
# [fx, fy, cx, cy] - Adjusting fx/fy changes depth perception
CAMERA_PARAMS = [600, 600, 320, 240] 
SHOOTER_VELOCITY = 16.54 # m/s
HUB_HEIGHT = 2.1 

# Initialize Mapper (Re-run this cell to apply Class changes)
mapper = FieldMapper(JSON_FILENAME)

# --- Calculate Hub Center ---
try:
    p2 = (mapper.tag_poses[2][0,3], mapper.tag_poses[2][1,3])
    p5 = (mapper.tag_poses[5][0,3], mapper.tag_poses[5][1,3])
    p4 = (mapper.tag_poses[4][0,3], mapper.tag_poses[4][1,3])
    p10 = (mapper.tag_poses[10][0,3], mapper.tag_poses[10][1,3])
    
    hub_center = get_intersection(p2, p5, p4, p10)
    print(f"✅ Hub Center calculated at: {hub_center}")
except KeyError:
    print("❌ Error: Could not find required tags (2,5,4,10) in JSON.")
    hub_center = (0,0)

In [ ]:
cap = cv2.VideoCapture(0)
detector = Detector(families='tag36h11')

plt.ion()
fig, ax = plt.subplots(figsize=(14, 8))

# 1. Draw Field Boundary
ax.plot([0, mapper.field_length, mapper.field_length, 0, 0], 
        [0, 0, mapper.field_width, mapper.field_width, 0], 'k-', linewidth=2)

# 2. Draw Tags & Orientation Arrows
tag_ids, tag_x, tag_y = [], [], []
tag_u, tag_v = [], []

for tid, pose in mapper.tag_poses.items():
    tag_ids.append(tid)
    tag_x.append(pose[0,3])
    tag_y.append(pose[1,3])
    R = pose[:3,:3]
    tag_u.append(R[0,0])
    tag_v.append(R[1,0])

ax.scatter(tag_x, tag_y, c='blue', marker='s', s=100, label='Tags')
ax.quiver(tag_x, tag_y, tag_u, tag_v, color='deepskyblue', scale=20, width=0.005)

# --- LABELS ---
for i, tid in enumerate(tag_ids):
    label_x = tag_x[i] + (tag_u[i] * 0.4)
    label_y = tag_y[i] + (tag_v[i] * 0.4)
    ax.text(label_x, label_y, str(tid), 
            ha='center', va='center',
            fontsize=8, fontweight='bold', color='white',
            bbox=dict(boxstyle="circle,pad=0.2", fc="navy", alpha=0.7, ec="none"),
            zorder=20)

# 3. Draw Hub Center
ax.plot(hub_center[0], hub_center[1], 'rx', markersize=15, markeredgewidth=3, label="Hub Center")

# 4. Setup Dynamic Robot Elements
robot_arrow = ax.quiver(0, 0, 1, 0, color='red', scale=25, label='Robot', zorder=10)
aim_line, = ax.plot([], [], 'g--', linewidth=2, label='Shooter Aim')

info_text = ax.text(0.5, 0.95, "Waiting for tags...", transform=ax.transAxes, 
                    ha="center", fontsize=12, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

ax.set_title("Robot Locator & Aimer (Heading Fixed)")
ax.axis('equal')
ax.grid(True)
ax.legend(loc='lower left')

print("Starting Loop... Press 'q' in CV2 window to stop.")

try:
    while True:
        ret, frame = cap.read()
        if not ret: break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        tags = detector.detect(gray, estimate_tag_pose=True, 
                               camera_params=CAMERA_PARAMS, tag_size=TAG_SIZE)
        
        x_estimates = []
        y_estimates = []
        yaw_vectors = []
        
        for tag in tags:
            corners = tag.corners.astype(int)
            for i in range(4):
                cv2.line(frame, tuple(corners[i]), tuple(corners[(i+1)%4]), (0, 255, 0), 2)
                
            res = mapper.get_cam_position(tag.tag_id, tag.pose_R, tag.pose_t)
            if res:
                rx, ry, _, ryaw = res
                x_estimates.append(rx)
                y_estimates.append(ry)
                yaw_vectors.append((np.cos(ryaw), np.sin(ryaw)))

        if len(x_estimates) > 0:
            final_x = sum(x_estimates) / len(x_estimates)
            final_y = sum(y_estimates) / len(y_estimates)
            
            sum_cos = sum(v[0] for v in yaw_vectors)
            sum_sin = sum(v[1] for v in yaw_vectors)
            final_yaw = np.arctan2(sum_sin, sum_cos)
            
            robot_arrow.set_offsets([final_x, final_y])
            robot_arrow.set_UVC(np.cos(final_yaw), np.sin(final_yaw))
            
            # Aiming Logic
            dist_to_hub = math.hypot(hub_center[0] - final_x, hub_center[1] - final_y)
            needed_yaw = math.atan2(hub_center[1] - final_y, hub_center[0] - final_x)
            pitch_angle = calculate_shot_angle_optimized(
                v0=SHOOTER_VELOCITY, distance=dist_to_hub, target_height=HUB_HEIGHT, arc="low"
            )
            
            aim_line.set_data([final_x, hub_center[0]], [final_y, hub_center[1]])
            
            status_str = f"DIST: {dist_to_hub:.2f}m | HEAD: {math.degrees(needed_yaw):.1f}°"
            if pitch_angle:
                status_str += f" | PITCH: {pitch_angle:.2f}°"
            else:
                status_str += " | ❌ OUT OF RANGE"
            
            info_text.set_text(status_str)
            plt.pause(0.01)

        cv2.imshow('Robot Camera', frame)
        if cv2.waitKey(1) == ord('q'):
            break

except KeyboardInterrupt:
    print("Stopped.")
finally:
    cap.release()
    cv2.destroyAllWindows()
    plt.close('all')